# Analise exploratória dos microdados do Enem 2023

In [58]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
import textwrap
import warnings

pd.set_option('display.max_rows', None)

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

In [59]:
file_path = '../data/raw/MICRODADOS_ENEM_2023.csv'
# [Acessar fonte](https://download.inep.gov.br/microdados/microdados_enem_2023.zip) 

df_amostra = pd.read_csv(file_path, sep=';', encoding='latin1')
# aproximadamente 2m30s para rodar o dataset inteiro

#df_amostra = pd.read_csv(file_path, sep=';', encoding='latin1', nrows=10000)

In [60]:
df_amostra.head(20)

,NU_INSCRICAO,NU_ANO,TP_FAIXA_ETARIA,TP_SEXO,TP_ESTADO_CIVIL,TP_COR_RACA,TP_NACIONALIDADE,TP_ST_CONCLUSAO,TP_ANO_CONCLUIU,TP_ESCOLA,...,Q016,Q017,Q018,Q019,Q020,Q021,Q022,Q023,Q024,Q025
0,210059085136,2023,14,M,2,1,1,1,17,1,...,C,C,B,B,A,B,B,A,A,B
1,210059527735,2023,12,M,2,1,0,1,16,1,...,B,A,B,B,A,A,C,A,D,B
2,210061103945,2023,6,F,1,1,1,1,0,1,...,B,A,A,B,A,A,A,A,A,B
3,210060214087,2023,2,F,1,3,1,2,0,2,...,A,A,A,B,A,A,D,A,A,B
4,210059980948,2023,3,F,1,3,1,2,0,2,...,A,A,A,B,A,A,B,A,A,A
5,210058061539,2023,6,F,1,3,1,1,0,1,...,B,A,A,B,A,A,C,A,A,B
6,210059855122,2023,11,F,1,3,1,1,12,1,...,B,A,A,B,A,B,B,A,A,B
7,210058387333,2023,11,M,1,3,1,1,12,1,...,B,A,A,A,A,A,B,A,B,B
8,210059085137,2023,5,F,1,2,1,1,1,1,...,B,A,A,B,A,A,C,A,A,B
9,210060801601,2023,11,M,1,1,1,1,8,1,...,B,A,B,C,B,A,C,A,B,B


## Especificações do dataset
- Estudar dimensionalidade do dataset
- Estudar preliminarmente a natureza das features
- Buscar por valores vazios

In [61]:
print(f"O dataset possui {df_amostra.shape[0]} linhas e {df_amostra.shape[1]} colunas.\n")

O dataset possui 3933955 linhas e 76 colunas.



In [62]:
df_amostra.dtypes.value_counts()

str        37
float64    21
int64      18
Name: count, dtype: int64

In [63]:
df_amostra.info()

<class 'pandas.DataFrame'>
RangeIndex: 3933955 entries, 0 to 3933954
Data columns (total 76 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   NU_INSCRICAO            int64  
 1   NU_ANO                  int64  
 2   TP_FAIXA_ETARIA         int64  
 3   TP_SEXO                 str    
 4   TP_ESTADO_CIVIL         int64  
 5   TP_COR_RACA             int64  
 6   TP_NACIONALIDADE        int64  
 7   TP_ST_CONCLUSAO         int64  
 8   TP_ANO_CONCLUIU         int64  
 9   TP_ESCOLA               int64  
 10  TP_ENSINO               float64
 11  IN_TREINEIRO            int64  
 12  CO_MUNICIPIO_ESC        float64
 13  NO_MUNICIPIO_ESC        str    
 14  CO_UF_ESC               float64
 15  SG_UF_ESC               str    
 16  TP_DEPENDENCIA_ADM_ESC  float64
 17  TP_LOCALIZACAO_ESC      float64
 18  TP_SIT_FUNC_ESC         float64
 19  CO_MUNICIPIO_PROVA      int64  
 20  NO_MUNICIPIO_PROVA      str    
 21  CO_UF_PROVA             int64  
 22  SG_UF

In [64]:
nulos = df_amostra.isnull().sum().sort_values(ascending=False)
percentual = (nulos / len(df_amostra)) * 100

pd.DataFrame({
    "nulos": nulos,
    "%": percentual
})

,nulos,%
SG_UF_ESC,2975449,75.635054
CO_UF_ESC,2975449,75.635054
NO_MUNICIPIO_ESC,2975449,75.635054
CO_MUNICIPIO_ESC,2975449,75.635054
TP_LOCALIZACAO_ESC,2975449,75.635054
TP_SIT_FUNC_ESC,2975449,75.635054
TP_DEPENDENCIA_ADM_ESC,2975449,75.635054
TP_ENSINO,2594874,65.960948
CO_PROVA_CN,1241528,31.559283
TX_GABARITO_MT,1241528,31.559283


## Construção do Dicionário de Dados

Será conduzido uma analise acerda das características de cada feature do dataset. 

O arquivo `resources/Dicionário_Microdados_Enem_2023.xlsx` representa o dicionário de dados oficial publicado juntamente ao dataset. A contrução do dicionário de dados neste nootebook será guiado por meio do dicionário oficial.

In [65]:
dados_dicionario = []

for coluna in df_amostra.columns:
    tipo = df_amostra[coluna].dtype
    valores_unicos = df_amostra[coluna].dropna().unique()
    
    # Pega até 5 exemplos de valores distintos
    exemplos = [str(v) for v in valores_unicos[:5]]
    
    dados_dicionario.append({
        'Nome da Variável': coluna,
        'Tipo de Dado': str(tipo),
        'Qtd Nulos': df_amostra[coluna].isnull().sum(),
        'Total Únicos': len(valores_unicos),
        'Exemplos Práticos': " | ".join(exemplos)
    })

df_dicionario = pd.DataFrame(dados_dicionario)

pd.set_option('display.max_rows', 150)
df_dicionario

,Nome da Variável,Tipo de Dado,Qtd Nulos,Total Únicos,Exemplos Práticos
0,NU_INSCRICAO,int64,0,3933955,210059085136 | 210059527735 | 210061103945 | 2...
1,NU_ANO,int64,0,1,2023
2,TP_FAIXA_ETARIA,int64,0,20,14 | 12 | 6 | 2 | 3
3,TP_SEXO,str,0,2,M | F
4,TP_ESTADO_CIVIL,int64,0,5,2 | 1 | 0 | 3 | 4
5,TP_COR_RACA,int64,0,6,1 | 3 | 2 | 0 | 5
6,TP_NACIONALIDADE,int64,0,5,1 | 0 | 4 | 2 | 3
7,TP_ST_CONCLUSAO,int64,0,4,1 | 2 | 3 | 4
8,TP_ANO_CONCLUIU,int64,0,18,17 | 16 | 0 | 12 | 1
9,TP_ESCOLA,int64,0,3,1 | 2 | 3


Após comparar o dicionário acima com o dicionário oficial, não foi encontrado divergências.

## Limpeza inicial dos dados
Antes de analisar mais de perto as fetures do dataset, será feito uma limpeza de dados inicial, dado os objetivos deste experimento e os insights propiciados pelo estudo inicial das colunas.

In [68]:
df = df_amostra.copy()

In [69]:
# exclusão das instâncias que representam candidados treineiros
n_treineiros = len(df[df['IN_TREINEIRO'] == 1])

df = df[df['IN_TREINEIRO'] == 0]
print("Instancias deletadas: ", n_treineiros)

Instancias deletadas:  620067


In [70]:
# exclusão das instâncias que representam candidados que faltaram em 1 ou mais provas
colunas_presenca = ['TP_PRESENCA_CN', 'TP_PRESENCA_CH', 'TP_PRESENCA_LC', 'TP_PRESENCA_MT']
n_ausentes = len(df[(df[colunas_presenca] != 1).any(axis=1)])

df = df[(df[colunas_presenca] == 1).all(axis=1)]
print("Instancias deletadas: ", n_ausentes)

Instancias deletadas:  1147045


In [71]:
colunas_excluir = [
# colunas com alta porcentagem de valores nulos (>65%)
"SG_UF_ESC", "CO_UF_ESC", "NO_MUNICIPIO_ESC", "CO_MUNICIPIO_ESC", "TP_LOCALIZACAO_ESC", "TP_SIT_FUNC_ESC", "TP_DEPENDENCIA_ADM_ESC", "TP_ENSINO",

# informações não relevantes para a analise
"NU_INSCRICAO", "IN_TREINEIRO",  "TP_SIT_FUNC_ESC", "TP_PRESENCA_CN", "TP_PRESENCA_CH", "TP_PRESENCA_LC", "TP_PRESENCA_MT", "CO_PROVA_CN", "CO_PROVA_CH", "CO_PROVA_LC", "CO_PROVA_MT", "NO_MUNICIPIO_PROVA", "CO_UF_PROVA",

# respostas do canditado e gabatito (uma vez que existe a nota calculada do candidato na base)
"TX_RESPOSTAS_CN", "TX_RESPOSTAS_CH", "TX_RESPOSTAS_LC", "TX_RESPOSTAS_MT", "TX_GABARITO_CN", "TX_GABARITO_CH", "TX_GABARITO_LC", "TX_GABARITO_MT",

# detalhes sobre a avaliação da redação 
# (serão removidos pois o experimento visa analisar de forma macro o desempenho do estudante, sendo apenas a nota final da redação o suficiente)
"TP_STATUS_REDACAO", "NU_NOTA_COMP1", "NU_NOTA_COMP2", "NU_NOTA_COMP3", "NU_NOTA_COMP4", "NU_NOTA_COMP5",
]

df = df.drop(colunas_excluir, axis=1)

In [72]:
df.columns

Index(['NU_ANO', 'TP_FAIXA_ETARIA', 'TP_SEXO', 'TP_ESTADO_CIVIL',
       'TP_COR_RACA', 'TP_NACIONALIDADE', 'TP_ST_CONCLUSAO', 'TP_ANO_CONCLUIU',
       'TP_ESCOLA', 'CO_MUNICIPIO_PROVA', 'SG_UF_PROVA', 'NU_NOTA_CN',
       'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'TP_LINGUA',
       'NU_NOTA_REDACAO', 'Q001', 'Q002', 'Q003', 'Q004', 'Q005', 'Q006',
       'Q007', 'Q008', 'Q009', 'Q010', 'Q011', 'Q012', 'Q013', 'Q014', 'Q015',
       'Q016', 'Q017', 'Q018', 'Q019', 'Q020', 'Q021', 'Q022', 'Q023', 'Q024',
       'Q025'],
      dtype='str')

In [ ]:
total_linhas_com_nulos = df.isnull().any(axis=1).sum()

print(f"Total de instâncias com pelo menos um valor nulo: {total_linhas_com_nulos}")

Total de instâncias com pelo menos um valor nulo: 0
Isso representa 0.00% do dataset.


In [77]:
print(f"O dataset original possui {df_amostra.shape[0]} linhas e {df_amostra.shape[1]} colunas.\n")
print(f"O dataset modificado possui {df.shape[0]} linhas e {df.shape[1]} colunas.\n")

O dataset original possui 3933955 linhas e 76 colunas.

O dataset modificado possui 2166843 linhas e 42 colunas.



In [ ]:
df.to_csv('..data/processed/df_limpo.csv', index=False)